In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"]
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

In [ ]:
## paper data
img_df = pd.read_csv("results/fig5_photos_va_results_paper.csv")
emo_df = pd.read_csv("results/fig3_cluster_results_paper.csv")
fig_name = "results_fig/fig5_paper.png"

## results from a rerun using Japanese prompts
#img_df = pd.read_csv("results/fig5_photos_va_results_test_jp.csv")
#emo_df = pd.read_csv("results/fig3_cluster_results_test_jp.csv")
#fig_name = "results_fig/fig5_test_jp.png"

## results from a rerun using English prompts
#img_df = pd.read_csv("results/fig5_photos_va_results_test_en.csv")
#emo_df = pd.read_csv("results/fig3_cluster_results_test_en.csv")
#fig_name = "results_fig/fig5_test_en.png"

In [ ]:
# visualize

img_df["label"] = img_df["label"].astype(int)
emo_df["ID"] = emo_df["ID"].astype(int)

img_df_avg = (
    img_df
    .groupby(["label", "image_name"], as_index=False)[["valence", "arousal"]]
    .mean()
)

all_labels = sorted(emo_df["ID"])

nrows, ncols = 11, 7
plots_per_fig = nrows * ncols

n_pages = math.ceil(len(all_labels) / plots_per_fig)

for page in range(n_pages):
    start = page * plots_per_fig
    end = start + plots_per_fig
    page_labels = all_labels[start:end]

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(10.5, 15),
        sharex=True,
        sharey=True
    )
    axes = axes.flatten()

    ticks = np.arange(-1.0, 1.01, 0.5)

    for ax, lab in zip(axes, page_labels):
        sub_img = img_df_avg[img_df_avg["label"] == lab]
        sub_emo = emo_df[emo_df["ID"] == lab]

        # points for each image
        ax.scatter(
            sub_img["valence"],
            sub_img["arousal"],
            alpha=0.7,
            s=18,
            label="images"
        )

        # point for the emotion label
        ax.scatter(
            sub_emo["valence"],
            sub_emo["arousal"],
            s=40,
            marker="x",
            linewidths=1.5
        )

        ax.set_title(f"ID {lab}", fontsize=12)
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)
        ax.tick_params(axis="both", labelsize=10)
        ax.axhline(0, color="gray", linewidth=0.7)
        ax.axvline(0, color="gray", linewidth=0.7)
        ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.4)
        ax.tick_params(labelsize=7, labelbottom=True, labelleft=True)

    # add legend
    if len(page_labels) < len(axes):
        legend_ax = axes[len(page_labels)]
        legend_ax.axis("off")

        handles = [
            plt.Line2D([], [], linestyle="none", marker="o", markerfacecolor="tab:blue", markersize=5, label="Facial images"),
            plt.Line2D([], [], linestyle="none", marker="x", color="tab:orange", markersize=7, label="Emotion label"),
        ]
        legend_ax.legend(
            handles=handles,
            loc="center",
            fontsize=10,
            frameon=False
        )

        for ax in axes[len(page_labels)+1:]:
            ax.axis("off")

    fig.supxlabel("Valence", fontsize=14)
    fig.supylabel("Arousal", fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.savefig(fig_name, dpi=600, bbox_inches="tight")
    plt.show()